# Final model: the team blend with the IBM Granite text score

**Decision (team, 2026-11-08):** the submitted ranking uses the IBM Granite text score as a model feature. It's variant **C** from `trackb_6_stacking.ipynb`: the team's logistic regression + LightGBM blend, plus two features:

- **`text_pct`:** the Granite text score's percentile within the CVE's snapshot month. The score comes from frozen `granite-embedding-small-english-r2` embeddings and a logistic regression; see `trackb_5_frozen_handoff.ipynb`.
- **`id_age`:** the publish year minus the year in the CVE ID. It lets the model discount old bugs re-filed under new IDs, which the text alone over-rates.

**The evidence, stated honestly** (`trackb_6_stacking.ipynb`, unchanged here):

| Test | Team model | With Granite (this model) | Difference |
|---|---|---|---|
| Train 2024, test 2025 (top 243 of 52) | 18 | **19** | +1 (90% range +0 to +3) |
| Repeated cross-validation (top 442 of 85) | 35.6 | **36.4** | +3 (90% range −1 to +7) |

The gain points the right way in both tests but is within noise. It was chosen for the Granite text signal, which carries information the other features lack (see the report), not because it's proven better.

**No leakage from the text score:** each training row's text score comes from a Granite model that never saw that row's label (out-of-fold), and 2026 is scored with a model trained only on 2018-2025.

## 1. Data and features (the team's code, unchanged)

In [1]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = next(p for p in [Path(".."), Path(".")] if (p / "ml" / "config.py").exists()).resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ml import data
from ml.featureset import FeatureBuilder

con = data.connect()
tr_rows, va_rows = data.load_splits(con)
fb = FeatureBuilder(con)
import features as F
L = pd.concat([fb.transform(tr_rows).assign(year=2024), fb.transform(va_rows).assign(year=2025)], ignore_index=True)
L = L.sort_values("cve_id").reset_index(drop=True)
sb = fb.transform(F.scoreboard_frame(con).sort_values("cve_id").reset_index(drop=True))
print(f"training rows (2024 + 2025): {len(L):,}, {int(L.y.sum())} exploited | 2026 scoreboard: {len(sb):,}")

training rows (2024 + 2025): 64,120, 85 exploited | 2026 scoreboard: 72,670


## 2. Add the Granite text score and ID age

The **final** text files are used: `text_oof_final.csv` for the 2024-2025 training rows (out-of-fold over 2018-2025), and `text_2026.csv` for the scoreboard (a model trained on all of 2018-2025). Scores from different text models are put on one scale as a percentile within each snapshot month.

In [2]:
TEXT = ROOT / "data" / "trackb" / "handoff_frozen"
def add_text(d, file):
    t = pd.read_csv(TEXT / file)
    d = d.merge(t, on="cve_id", how="left")
    assert d.text_score.notna().all(), f"{d.text_score.isna().sum()} rows have no text score"
    d["text_pct"] = d.groupby("snap")["text_score"].rank(pct=True)
    d["id_age"] = (pd.to_datetime(d["published"]).dt.year - d["cve_id"].str.slice(4, 8).astype(int)).clip(0, 20)
    return d
L, sb = add_text(L, "text_oof_final.csv"), add_text(sb, "text_2026.csv")
print("text score coverage: training 100%, 2026 100%")

text score coverage: training 100%, 2026 100%


## 3. Train on 2024 + 2025 and score 2026

These are the team's settings, unchanged: logistic regression (`C` = 0.1, balanced class weights) and LightGBM (200 depth-2 trees, averaged over 30 seeds), blended 50/50 by rank.

In [3]:
BASE = ["any_exploit", "poc_github", "nuclei", "log_assigner_kev_rate", "kw_unauth", "pr_none", "ui_none", "cvss_filled", "cwe_auth", "cwe_path"]
NEW4 = ["net_preauth", "log_batch_size", "log_cwe_kev_rate", "product_has_kev"]
GRANITE = ["text_pct", "id_age"]
GBM = dict(n_estimators=200, learning_rate=0.03, num_leaves=4, max_depth=2, min_child_samples=40, subsample=0.8, subsample_freq=1,
           colsample_bytree=0.8, reg_lambda=5, scale_pos_weight=50, verbose=-1)

def final_scores(extra):
    c_lr, c_gb = BASE + extra, BASE + NEW4 + extra
    lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)).fit(L[c_lr], L.y)
    p_lr = lr.predict_proba(sb[c_lr])[:, 1]
    gbms = []
    for s in range(30):
        perm = np.random.default_rng(s).permutation(len(L))
        gbms.append(lgb.LGBMClassifier(**GBM, random_state=s).fit(L.iloc[perm][c_gb], L.iloc[perm].y))
    p_gb = np.mean([m.predict_proba(sb[c_gb])[:, 1] for m in gbms], axis=0)
    return 0.5 * rankdata(p_lr) / len(sb) + 0.5 * rankdata(p_gb) / len(sb), lr, gbms, c_lr, c_gb

score, lr, gbms, c_lr, c_gb = final_scores(GRANITE)
score_team, *_ = final_scores([])
top = sb.assign(score=score).nlargest(500, "score")
top_team = sb.assign(score=score_team).nlargest(500, "score")
print(f"Final model: LR {len(c_lr)} features, LightGBM {len(c_gb)} features")

Final model: LR 12 features, LightGBM 16 features


## 4. What the Granite features change

In [4]:
w = pd.Series(lr[-1].coef_[0], index=c_lr).sort_values(ascending=False)
gain = pd.Series(np.mean([m.booster_.feature_importance("gain") for m in gbms], axis=0), index=c_gb)
gain = (gain / gain.sum()).sort_values(ascending=False)
print("LR weight (standardized):", {k: round(v, 2) for k, v in w.items() if k in GRANITE})
print("LightGBM gain share:", {k: f"{v:.1%}" for k, v in gain.items() if k in GRANITE},
      f"(rank {list(gain.index).index('text_pct') + 1} of {len(gain)} for text_pct)")
overlap = top.cve_id.isin(top_team.cve_id).sum()
print(f"\nTop 500 overlap with the no-text team model: {overlap} of 500 ({500 - overlap} CVEs swapped)")
comp = pd.DataFrame({"with Granite": [top.any_exploit.mean(), (top.id_age >= 2).sum(), (top.assigner == 'disclosure@vulncheck.com').sum(), top.text_pct.median()],
                     "team model (no text)": [top_team.any_exploit.mean(), (top_team.id_age >= 2).sum(), (top_team.assigner == 'disclosure@vulncheck.com').sum(), top_team.text_pct.median()]},
                    index=["share with public exploit code", "old-ID CVEs", "VulnCheck-issued CVEs", "median text percentile"])
comp.round(3)

LR weight (standardized): {'text_pct': 0.61, 'id_age': -0.5}
LightGBM gain share: {'text_pct': '23.6%', 'id_age': '0.2%'} (rank 2 of 16 for text_pct)

Top 500 overlap with the no-text team model: 390 of 500 (110 CVEs swapped)


,with Granite,team model (no text)
share with public exploit code,0.862,0.872
old-ID CVEs,2.000,5.000
VulnCheck-issued CVEs,64.000,69.000
median text percentile,0.895,0.827


**Reading this:**
- **Granite is now a real part of the model.** `text_pct` is LightGBM's **second most important feature** (23.6% of its splitting gain, after the issuer's KEV rate), and it gets a positive weight in logistic regression (+0.61). `id_age` gets a negative weight (−0.50), so the model discounts old bugs re-filed under new IDs.
- **It changes 110 of the 500 picks** (390 overlap with the no-text list).
- **The list keeps its character:** 86% of picks have public exploit code (87% before), and **old-ID CVEs fall from 5 to 2**. The fix for the text model's main weakness shows up in the list.

In [5]:
swapped_in = top[~top.cve_id.isin(top_team.cve_id)].sort_values("score", ascending=False)
print(f"{len(swapped_in)} CVEs enter the list because of the Granite features. Exploit code among them: {swapped_in.any_exploit.mean():.0%}; "
      f"median text percentile: {swapped_in.text_pct.median():.2f}")
swapped_in[["cve_id", "assigner", "any_exploit", "text_pct", "id_age", "description"]].head(8).assign(description=lambda d: d.description.str[:90])

110 CVEs enter the list because of the Granite features. Exploit code among them: 78%; median text percentile: 0.93


,cve_id,assigner,any_exploit,text_pct,id_age,description
69942,CVE-2026-9254,f23511db-6c3e-4e32-a477-6aa17d310630,1,0.993383,0,An unauthenticated OS command injection vulner...
46007,CVE-2026-57517,disclosure@vulncheck.com,1,0.988177,0,Control Web Panel before 0.9.8.1225 contains a...
51398,CVE-2026-64633,support@hackerone.com,1,0.992567,0,A vulnerability allowing remote unauthenticate...
19769,CVE-2026-22666,disclosure@vulncheck.com,1,0.998621,0,Dolibarr ERP/CRM versions prior to 23.0.2 cont...
23224,CVE-2026-26416,cve@mitre.org,1,0.944275,0,An authorization bypass vulnerability in Tata ...
37818,CVE-2026-46453,security@apache.org,1,0.996436,0,"Improper Input Validation, Authorization Bypas..."
54968,CVE-2026-70559,disclosure@vulncheck.com,1,0.946577,0,Dinky's SysConfigController.getAll() handler f...
60126,CVE-2026-77113,security@ubuntu.com,1,0.772668,0,Path traversal in apport-unpack in Canonical A...


**Reading this:** the 110 CVEs that enter the list have a median text percentile of 0.93, and 78% have public exploit code. They're CVEs where the description reads like a classic exploited bug (unauthenticated command injection, authorization bypass) *and* there's other evidence. The text tips close calls; it doesn't override the strongest signals.

## 5. Write the submission file

In [6]:
out = ROOT / "outputs"; out.mkdir(exist_ok=True)
assert len(top) == 500 and top.cve_id.is_unique and top.cve_id.isin(sb.cve_id).all()
top[["cve_id"]].to_csv(out / "top500.csv", index=False)
top_team[["cve_id"]].to_csv(out / "top500_without_text.csv", index=False)
chk = pd.read_csv(out / "top500.csv")
print(f"outputs/top500.csv: {len(chk)} rows, column {list(chk.columns)}, unique {chk.cve_id.is_unique}, all on the scoreboard {chk.cve_id.isin(sb.cve_id).all()}")
print("outputs/top500_without_text.csv kept for comparison")

outputs/top500.csv: 500 rows, column ['cve_id'], unique True, all on the scoreboard True
outputs/top500_without_text.csv kept for comparison
